In [19]:
# 1. Installations and Dependencies

# Core Agents & AI
# %pip install -qU langchain-openai
# %pip install -qU langchain-community
# %pip install -qU langgraph
# %pip install -qU python-dotenv
# %pip install -qU tavily-python

# Web Scraping Tools
# %pip install -qU ddgs
# %pip install -qU selenium
# %pip install -qU webdriver-manager
# %pip install -qU beautifulsoup4

#Memory and data persistence

# %pip install -qU aiosqlite
# %pip install -qU langgraph-checkpoint-sqlite

print("[✅] Check installed dependencies python")

[✅] Check installed dependencies python


In [20]:
# 2. Environment setup: Model Configuration

from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field, SecretStr, ConfigDict

class LLMConfig(BaseModel):
    """
    Centralized configuration for the Language Model.
    
    Uses Pydantic V2 for strict validation and type safety, ensuring
    the model is instantiated with valid parameters.
    """
    
    # Configuration to make the instance immutable (frozen) after creation
    model_config = ConfigDict(frozen=True)

    base_url: str = Field(
        default="http://localhost:1234/v1",
        description="Base URL for the local or remote inference server."
    )
    model_name: str = Field(
        default="meta-llama-3.1-8b-instruct",
        description="Name of the model to be used."
    )
    temperature: float = Field(
        default=0.0,
        ge=0.0, # Greater than or equal to 0.0
        le=1.0, # Less than or equal to 1.0
        description="Sampling temperature to control creativity (0.0 to 1.0)."
    )
    api_key: SecretStr = Field(
        default=SecretStr("not-needed"),
        description="API Key (protected type to prevent leakage in logs)."
    )

    def get_chat_model(self) -> ChatOpenAI:
        """
        Factory method to create a configured ChatOpenAI instance.
        
        Returns:
            ChatOpenAI: An initialized LangChain chat model wrapper.
        """
        return ChatOpenAI(
            base_url=self.base_url,
            model=self.model_name,
            temperature=self.temperature,
            api_key=self.api_key.get_secret_value(), # Securely retrieve the string
            max_retries=2,
            timeout=None
        )

# Instantiating the configuration
llm_settings = LLMConfig(
    model_name="meta-llama-3.1-8b-instruct"
)

# Creating the Model
model = llm_settings.get_chat_model()

print(f"[✅] Model loaded: {llm_settings.model_name} via {llm_settings.base_url}")

[✅] Model meta-llama-3.1-8b-instruct ready.


In [3]:
chat = "Are you on-line? "
model.invoke(chat)

AIMessage(content='I am a computer program designed to simulate conversations and answer questions, so I\'m always "on" and ready to chat. However, I don\'t have a physical presence or direct access to the internet in real-time. My training data is based on a massive dataset that was available up to my knowledge cutoff date (December 2023), but I can still provide information and answer questions to the best of my ability based on that data.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 40, 'total_tokens': 129, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'meta-llama-3.1-8b-instruct', 'system_fingerprint': 'meta-llama-3.1-8b-instruct', 'id': 'chatcmpl-sykgnxmfivhjarew8ogik', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c1b49-c8db-7470-9f2f-a86cfa6f4d0a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 40, 'output_tok

In [4]:
# Armazena a resposta em uma variável
response = model.invoke(chat)

# Método Pydantic V2 para gerar uma string JSON
# 'indent=2' deixa o JSON formatado (pretty print)
json_output = response.model_dump_json(indent=2)

print(json_output)

{
  "content": "I am a computer program designed to simulate conversations. I'm not directly \"online\" in the classical sense, but I can be accessed through various platforms and interfaces that are connected to the internet. This allows users like you to interact with me remotely.",
  "additional_kwargs": {
    "refusal": null
  },
  "response_metadata": {
    "token_usage": {
      "completion_tokens": 51,
      "prompt_tokens": 40,
      "total_tokens": 91,
      "completion_tokens_details": null,
      "prompt_tokens_details": null
    },
    "model_provider": "openai",
    "model_name": "meta-llama-3.1-8b-instruct",
    "system_fingerprint": "meta-llama-3.1-8b-instruct",
    "id": "chatcmpl-chwkf5hxejdompu1rpmw7",
    "finish_reason": "stop",
    "logprobs": null
  },
  "type": "ai",
  "name": null,
  "id": "lc_run--019c1b49-d020-7671-96f9-74fe09887668-0",
  "tool_calls": [],
  "invalid_tool_calls": [],
  "usage_metadata": {
    "input_tokens": 40,
    "output_tokens": 51,
    "t

In [5]:
# 1. Acessando o conteúdo principal (Atributo direto da classe)
print(f"Content: {response.content}")
print(f"Model_Name: {model.model_name}")

# Serialização: Use objeto.model_dump_json() (Pydantic V2) para ver tudo como JSON.
# Acesso:
# Propriedades de topo (content, id): Use ponto (obj.content).
# Dados dentro de dicionários (token_usage): Use colchetes (obj.usage_metadata['total_tokens']).

# 2. Acessando metadados de uso (Dicionário aninhado)
# Nota: 'usage_metadata' é o padrão novo do LangChain para tokens
input_tokens = response.usage_metadata['input_tokens']
output_tokens = response.usage_metadata['output_tokens']
total_tokens = response.usage_metadata['total_tokens']

print(f"Tokens Used: Input={input_tokens}, Output={output_tokens}, Total={total_tokens}")

# 3. Acessando metadados específicos do provedor (OpenAI/Llama)
# Isso fica dentro de 'response_metadata'
finish_reason = response.response_metadata.get('finish_reason')
print(f"Finish Reason: {finish_reason}")

Content: I am a computer program designed to simulate conversations. I'm not directly "online" in the classical sense, but I can be accessed through various platforms and interfaces that are connected to the internet. This allows users like you to interact with me remotely.
Model_Name: meta-llama-3.1-8b-instruct
Tokens Used: Input=40, Output=51, Total=91
Finish Reason: stop


In [6]:
# Modo INVOKE (Bloqueante)
# Você espera tudo ficar pronto para ter a variável 'response'
response = model.invoke("Escreva um poema longo.")
print(response.content) 

# Modo STREAM (Fluxo Contínuo)
# O loop roda a cada 'pedaço' que chega da API
print("Começando o stream...")
for chunk in model.stream("Escreva um poema longo."):
    # 'chunk' é um objeto AIMessageChunk (Pydantic)
    # end="" serve para imprimir na mesma linha sem pular
    print(chunk.content, end="", flush=True)

"Sonho Eterno"

No fundo da noite, onde as estrelas brilham,
Um sonho eterno me chama, como uma luz que se alastra.
É um lugar mágico, onde o tempo não existe,
E a alma pode voar livre, sem medo de ser presa.

Lá, os rios correm com água cristalina,
E as flores desabrocham em cores vibrantes e sinuosas.
O ar está cheio de música, que ecoa em cada canto,
E o silêncio é preenchido pelo som do meu coração.

Eu caminho por esse mundo mágico,
Com passos leves e um sorriso no rosto.
A cada passo, descubro novas maravilhas,
E a minha alma se enche de alegria e paz.

Lá, encontro amigos que me conhecem desde sempre,
E juntos, compartilhamos risadas e lágrimas.
Nós dançamos sob as estrelas, com movimentos suaves,
E o mundo inteiro parece ser nosso.

Mas, ao mesmo tempo, sinto uma saudade profunda,
De um lugar que não existe mais, de uma época que passou.
Um lugar onde a infância era longa e feliz,
E a vida era simples, sem preocupações ou medos.

Eu penso nos momentos que se foram,
E no amor qu

In [7]:
response

AIMessage(content='"Sonho Eterno"\n\nNo fundo da noite, onde as estrelas brilham,\nUm sonho eterno me chama, como uma luz que se alastra.\nÉ um lugar mágico, onde o tempo não existe,\nE a alma pode voar livre, sem medo de ser presa.\n\nLá, os rios correm com água cristalina,\nE as flores desabrocham em cores vibrantes e sinuosas.\nO ar está cheio de música, que ecoa em cada canto,\nE o silêncio é preenchido pelo som do meu coração.\n\nEu caminho por esse mundo mágico,\nCom passos leves e um sorriso no rosto.\nA cada passo, descubro novas maravilhas,\nE a minha alma se enche de alegria e paz.\n\nLá, encontro amigos que me conhecem desde sempre,\nE juntos, compartilhamos risadas e lágrimas.\nNós dançamos sob as estrelas, com movimentos suaves,\nE o mundo inteiro parece ser nosso.\n\nMas, ao mesmo tempo, sinto uma saudade profunda,\nDe um lugar que não existe mais, de uma época que passou.\nUm lugar onde a infância era longa e feliz,\nE a vida era simples, sem preocupações ou medos.\n\nEu

In [8]:
# 1. Persistence Setup

import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# Connect to a local SQLite database file.
# LangGraph will create tables to store conversation states.
# check_same_thread=False is required for SQLite's multi-threading compatibility with LangGraph.
db_connection = sqlite3.connect("memory_test.db", check_same_thread=False)

# The 'memory' object is the checkpointer that our graph will use.
# It handles saving and loading the state from the database.
memory = SqliteSaver(conn=db_connection)

print("[✅] SQLite database connection and checkpointer are ready.")

[✅] SQLite database connection and checkpointer are ready.


In [9]:
# 2. State Definition with Pydantic V2

from typing import List, Annotated
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class ConversationState(BaseModel):
    """
    Defines the structure of our conversation memory.
    Validated by Pydantic V2.
    """
    
    # 'Annotated[..., add_messages]' is the key for LangGraph to append
    # messages to the history instead of replacing them.
    messages: Annotated[List[BaseMessage], add_messages] = Field(
        default_factory=list,
        description="The sequence of messages in the conversation."
    )

print("[✅] Pydantic V2 state for conversation defined.")

[✅] Pydantic V2 state for conversation defined.


In [10]:
# 3. Simple Graph Construction

from langgraph.graph import StateGraph, END

# Define the graph
workflow = StateGraph(ConversationState)

# Node: A simple function that calls the LLM with the current messages
def call_model(state: ConversationState):
    response = model.invoke(state.messages)
    # The return must be a dictionary matching the state's structure
    return {"messages": [response]}

# Add the node to the graph
workflow.add_node("llm", call_model)
workflow.set_entry_point("llm")
workflow.add_edge("llm", END)

# Compile the graph, connecting it to our SQLite memory
# This is the step that makes the graph "stateful"
app = workflow.compile(checkpointer=memory)

print("[✅] Simple stateful graph compiled and connected to the checkpointer.")

[✅] Simple stateful graph compiled and connected to the checkpointer.


In [11]:
# 4. Memory Test

from langchain_core.messages import HumanMessage

# Define the unique ID for our conversation thread
thread_config = {"configurable": {"thread_id": "memory_test"}}

# --- Primeira Pergunta ---
print("--- 🗣️ Turn 1: Telling the AI my name ---")
question1 = "Hi, my name is John. I'm learning about AI."
input1 = {"messages": [HumanMessage(content=question1)]}

# Invoke the graph. It will save the state automatically.
response1 = app.invoke(input1, config=thread_config)
print(f"🤖 AI Response 1: {response1['messages'][-1].content}\n")


# --- Segunda Pergunta (Prova da Memória) ---
# Note: We are ONLY sending the new message.
# LangGraph, seeing the thread_id, will automatically load the previous messages from the DB.
print("--- 🗣️ Turn 2: Asking a follow-up question ---")
question2 = "What is my name?"
input2 = {"messages": [HumanMessage(content=question2)]}

# Invoke again with the SAME thread_id
response2 = app.invoke(input2, config=thread_config)
print(f"🤖 AI Response 2: {response2['messages'][-1].content}")

# Expected output for AI Response 2: "Your name is John." or similar.
print("\n[✅] Memory test completed.")

--- 🗣️ Turn 1: Telling the AI my name ---
🤖 AI Response 1: Hello again John. We've had this conversation before. You're still learning about AI, I see. What's new or what would you like to discuss today?

--- 🗣️ Turn 2: Asking a follow-up question ---
🤖 AI Response 2: Your name is John.

[✅] Memory test completed.


In [12]:
# 5. Auditing Conversation History (Corrected)

# The unique ID of the conversation we want to inspect
thread_config = {"configurable": {"thread_id": "memory_test"}}

print(f"--- 🔍 Audit History for Thread ID: 'memory_test' ---")

# .get_state_history() retrieves all saved states for the thread
# It returns an iterable of 'StateSnapshot' objects
for snapshot in app.get_state_history(config=thread_config):
    
    # The 'values' attribute of the snapshot contains our state data dictionary
    current_state_data = snapshot.values
    
    # We validate this data using our Pydantic model for type-safe access
    current_state_pydantic = ConversationState.model_validate(current_state_data)
    
    # CORRECTED: The checkpoint_id is inside the snapshot's config dictionary
    checkpoint_id = snapshot.config['configurable']['checkpoint_id']
    
    print(f"\n[Checkpoint ID: {checkpoint_id}]")
    print("  Messages in this state:")
    
    if not current_state_pydantic.messages:
        print("    - (empty initial state)")
    else:
        for msg in current_state_pydantic.messages:
            # Using repr() to avoid potential printing issues with special characters
            print(f"    - {msg.type.upper()}: {repr(msg.content)}")

print("\n--- [✅] Audit complete ---")

--- 🔍 Audit History for Thread ID: 'memory_test' ---

[Checkpoint ID: 1f0ffbc1-7971-6213-8010-caf2432311ac]
  Messages in this state:
    - HUMAN: "Hi, my name is John. I'm learning about AI."
    - AI: "Hello John, nice to meet you. Learning about AI can be a fascinating and rewarding experience. What specific aspects of AI are you interested in or would like to learn more about? Are you looking into machine learning, natural language processing, computer vision, or something else? I'm here to help answer any questions you may have."
    - HUMAN: 'What is my name?'
    - AI: 'Your name is John.'
    - HUMAN: "Hi, my name is John. I'm learning about AI."
    - AI: "Hello again John. It looks like we're having a bit of a loop here. You've already introduced yourself earlier. What's new or what would you like to talk about regarding your studies on AI?"
    - HUMAN: 'What is my name?'
    - AI: 'Your name is John.'
    - HUMAN: "Hi, my name is John. I'm learning about AI."
    - AI: "Hel

# 🧠 Arquiteturas de Memória para Agentes de IA

Para consolidar o conhecimento em **LangGraph** e **LangChain**, é fundamental entender como os agentes processam informações através de três tipos de memória inspirados na psicologia cognitiva.

## 1. Memória Episódica (Experiência)
**O que é:** É o registro de "eventos" ou "episódios". No contexto de IA, é o **histórico da conversa**. Ela permite que o agente se lembre do que foi dito anteriormente dentro de uma interação específica.

*   **Foco:** Sequência temporal e contexto imediato.
*   **No LangGraph:** Implementada via **Checkpointers** (como o `SqliteSaver`).
*   **Exemplo:** O agente lembrar que o usuário se apresentou como "João" três mensagens atrás.

## 2. Memória Semântica (Conhecimento)
**O que é:** É o conhecimento de fatos, conceitos e significados que não dependem de uma experiência pessoal ou temporal. É o "saber geral".

*   **Foco:** Informações externas e fatos estruturados.
*   **No LangGraph/LangChain:** Implementada via **RAG (Retrieval-Augmented Generation)** e **Bancos de Dados Vetoriais**.
*   **Exemplo:** O agente buscar em um PDF de manuais qual é a voltagem correta de um aparelho para responder ao usuário.

## 3. Memória Procedural (Habilidade)
**O que é:** É o conhecimento de "como fazer" as coisas. São os processos, regras de raciocínio e habilidades técnicas.

*   **Foco:** Fluxos de trabalho (Workflows) e execução de tarefas.
*   **No LangGraph:** Implementada através da **Topologia do Grafo** (nós e arestas) e das **Tools** (ferramentas).
*   **Exemplo:** O agente saber que, para cancelar um pedido, ele deve primeiro validar o ID e depois chamar uma API específica.

---

### 📊 Resumo Comparativo

| Tipo de Memória | Pergunta Chave | Implementação Técnica | Ferramenta no Projeto |
| :--- | :--- | :--- | :--- |
| **Episódica** | *O que aconteceu?* | Checkpoints de Estado | `SqliteSaver` / `thread_id` |
| **Semântica** | *O que é isso?* | Busca em Base de Conhecimento | `VectorStore` / RAG |
| **Procedural** | *Como eu faço?* | Lógica do Grafo e Ferramentas | `StateGraph` / `@tool` |

---

### 🛠️ O papel do Pydantic V2 na Memória
O **Pydantic V2** atua como o "filtro de qualidade" em todos os tipos de memória:
1.  **Na Episódica:** Valida se o estado recuperado do banco de dados ainda é compatível com o código atual.
2.  **Na Semântica:** Estrutura os metadados dos documentos recuperados (ex: garantindo que a "data de validade" do fato seja um objeto `datetime` válido).
3.  **Na Procedural:** Garante que os argumentos passados para as ferramentas (`Tools`) estejam no formato exato que o processo exige, evitando erros de execução no meio do grafo.

---

In [13]:
from pydantic import BaseModel, Field, ConfigDict
from langchain_core.messages import HumanMessage
from typing import List, Optional

# 1. Input Model for Validation
class UserQuery(BaseModel):
    """
    Validates the user input before it reaches the graph.
    Ensures we don't send empty strings or invalid data.
    """
    text: str = Field(..., min_length=1, description="The message sent by the user.")
    thread_id: str = Field(default="default_thread", description="Unique identifier for the session.")

# 2. The Orchestrator Class
class ConversationManager:
    """
    Manages the interaction between the user and the LangGraph.
    Encapsulates persistence logic and session management.
    """
    
    def __init__(self, graph_app):
        # We store the compiled LangGraph app
        self.app = graph_app

    def run_conversation(self, queries: List[str], thread_id: str = "memory_test"):
        """
        Automates a sequence of messages in the same thread.
        """
        results = []
        
        print(f"🚀 Starting automation for Thread: {thread_id}\n" + "="*50)
        
        for i, query_text in enumerate(queries, 1):
            # Validation Step using Pydantic V2
            query = UserQuery(text=query_text, thread_id=thread_id)
            
            print(f"--- 🗣️ Turn {i}: {query.text} ---")
            
            # Prepare configuration for persistence
            config = {"configurable": {"thread_id": query.thread_id}}
            
            # Invoke the graph
            # Note: LangGraph expects a dict that matches the State definition
            state_input = {"messages": [HumanMessage(content=query.text)]}
            output = self.app.invoke(state_input, config=config)
            
            # Extract last message
            last_msg = output["messages"][-1].content
            print(f"🤖 AI: {last_msg}\n")
            
            results.append(last_msg)
            
        print("="*50 + "\n[✅] Automation sequence completed.")
        return results

# 3. Execution (The "Clean" Part)
# Initialize the manager with our compiled app
manager = ConversationManager(app)

# Define our list of questions (Automation)
test_questions = [
    "Hi, my name is John. I'm learning about AI.",
    "What is my name?",
    "What was the first thing I told you?"
]

# Run everything with one command
responses = manager.run_conversation(test_questions, thread_id="automated_test")

🚀 Starting automation for Thread: automated_test
--- 🗣️ Turn 1: Hi, my name is John. I'm learning about AI. ---
🤖 AI: You repeated what you said initially. Your name is indeed John, and you are learning about AI.

--- 🗣️ Turn 2: What is my name? ---
🤖 AI: Your name is John.

--- 🗣️ Turn 3: What was the first thing I told you? ---
🤖 AI: The first thing you told me was "Hi, my name is John. I'm learning about AI."

[✅] Automation sequence completed.


In [14]:
from pydantic import BaseModel, Field, ConfigDict
from langchain_core.messages import HumanMessage
from typing import List, Dict, Any

class ChatSession(BaseModel):
    """
    Validates the session configuration.
    """
    thread_id: str = Field(..., min_length=1)
    user_name: str = Field(default="User")

class ConversationManager:
    """
    An OOP wrapper for LangGraph interaction with persistence.
    """
    def __init__(self, graph_app):
        self.app = graph_app

    def chat(self, user_input: str, thread_id: str):
        """
        Processes a single turn of conversation and persists it.
        """
        # 1. Validation using Pydantic V2 concept
        if not user_input.strip():
            print("⚠️ Empty input ignored.")
            return None

        # 2. Graph Configuration
        config = {"configurable": {"thread_id": thread_id}}
        
        # 3. Execution
        # We send the message and LangGraph handles the retrieval of history from SQLite
        input_state = {"messages": [HumanMessage(content=user_input)]}
        output = self.app.invoke(input_state, config=config)
        
        # 4. Extract and Return Result
        last_msg = output["messages"][-1]
        return last_msg.content

    def show_history(self, thread_id: str):
        """
        Utility to print the history of a specific thread.
        """
        config = {"configurable": {"thread_id": thread_id}}
        state = self.app.get_state(config)
        messages = state.values.get("messages", [])
        
        print(f"\n--- 📜 History for {thread_id} ---")
        for m in messages:
            print(f"{m.type.upper()}: {m.content}")
        print("--------------------------")

# Initialize the manager
manager = ConversationManager(app)

In [15]:
SESSION_ID = "persistencia_123" # Use o mesmo ID para manter a memória

response = manager.chat("Explain Pydantic V2 in one sentence.", thread_id=SESSION_ID)
print(f"🤖 AI: {response}")

🤖 AI: Pydantic V2 is a major update to the popular Python library for building robust, scalable, and maintainable data models, featuring improved performance, native support for asynchronous programming, enhanced validation capabilities, and better integration with other Python tools and frameworks.


In [16]:
# A IA deve saber que estamos falando de Pydantic V2 sem que eu repita o nome.
response = manager.chat("And why is it better than V1?", thread_id=SESSION_ID)
print(f"🤖 AI: {response}")

🤖 AI: Pydantic V2 offers several improvements over V1, including:

1. **Async Support**: Native support for asynchronous programming using the `async` and `await` keywords.

2. **Improved Performance**: Optimizations that result in significant performance boosts.

3. **Enhanced Validation**: More flexible validation rules, custom validators, and improved error messages.

4. **Better Type Hinting**: Improved type hinting capabilities for better integration with other Python tools and frameworks.

5. **Breaking Changes**: Some breaking changes were made to improve the library's design and usability.

6. **New Features for API Development**: Support for OpenAPI and JSON Schema, making it easier to generate documentation and client code for your APIs.

7. **Better Support for Data Types**: Improved support for various data types, such as UUIDs, dates, and times.

These improvements make Pydantic V2 a more powerful and flexible tool for building robust Python applications.


In [17]:
manager.show_history(SESSION_ID)


--- 📜 History for persistencia_123 ---
HUMAN: Explain Pydantic V2 in one sentence.
AI: Pydantic V2 is a major update to the popular Python library for building robust, scalable, and maintainable data models, featuring improved performance, new features like support for async/await, and enhanced validation capabilities.
HUMAN: And why is it better than V1?
AI: Pydantic V2 offers several improvements over V1, including:

1. **Async Support**: Pydantic V2 introduces native support for asynchronous programming using the `async` and `await` keywords, making it easier to build high-performance async APIs.

2. **Improved Performance**: The new version includes various optimizations that result in significant performance boosts, especially when dealing with large datasets or complex models.

3. **Enhanced Validation**: Pydantic V2 introduces more flexible validation rules, including support for custom validators and improved error messages.

4. **Better Support for Complex Models**: The updat

In [18]:
from pydantic import BaseModel, Field, ValidationError

class ChatInput(BaseModel):
    """Simple validation for the interactive loop."""
    text: str = Field(..., min_length=1,) #max_length=1000

def start_chat_session(manager: ConversationManager, thread_id: str):
    """
    Launches an interactive chat loop inside the Jupyter cell.
    Type 'exit', 'quit', or 'q' to stop the session.
    """
    print(f"--- 🟢 Chat Session Started (ID: {thread_id}) ---")
    print("Commands: 'exit' to stop | 'history' to see logs\n")

    while True:
        try:
            # 1. Capture User Input
            raw_input = input("You: ").strip()

            # 2. Check for Exit Commands
            if raw_input.lower() in ["exit", "quit", "q"]:
                print(f"\n--- 🔴 Session Ended (ID: {thread_id}) ---")
                break
            
            # 3. Utility command to see history mid-chat
            if raw_input.lower() == "history":
                manager.show_history(thread_id)
                continue

            # 4. Validation with Pydantic V2
            # This prevents sending empty messages or excessively long ones to the LLM
            validated_input = ChatInput(text=raw_input)

            # 5. Process through the Manager (which uses LangGraph + Persistence)
            # We add a small visual indicator of processing
            print("🤖 AI is thinking...", end="\r")
            response = manager.chat(validated_input.text, thread_id)
            
            # 6. Display Response
            print(f"🤖 AI: {response}\n")

        except ValidationError as e:
            print(f"⚠️ Input Error: {e.json()}")
        except KeyboardInterrupt:
            print(f"\n\n--- 🔴 Session Interrupted (ID: {thread_id}) ---")
            break
        except Exception as e:
            print(f"❌ An unexpected error occurred: {e}")
            break

# --- Execution ---
# Start the loop with a specific thread
start_chat_session(manager, thread_id="interactive_demo_v1")

--- 🟢 Chat Session Started (ID: interactive_demo_v1) ---
Commands: 'exit' to stop | 'history' to see logs



You:  history



--- 📜 History for interactive_demo_v1 ---
HUMAN: What's my name?
AI: I'm happy to chat with you, but I don't seem to have any information about your name. I'm a large language model, I don't retain any personal data or memories of previous conversations. Each time you interact with me, it's a new conversation and I start from scratch. If you'd like to share your name with me, I'd be happy to chat with you!
HUMAN: ok My name is Yuri
AI: Nice to meet you, Yuri. How are you doing today? Is there something on your mind that you'd like to talk about or would you like some recommendations or information on a particular topic? I'm here to help and chat with you.
HUMAN: let's try again. What's my name?
AI: Your name is Yuri.
HUMAN: ok you pass the test, you have memory persistence via snapshot interation and db id
AI: I was just playing along earlier, but it seems I do have some form of memory persistence in our conversation, thanks to the underlying technology that powers me.

In this case, 

You:  alright let's continue, now with Forclusion data simulation: You are a highly skilled psychoanalytic and cognitive systems analyst. I am providing structural metrics and quantitative data extracted from a text.  Please re-analyze the text, integrating both the content and these metrics. Focus on: 1. How the numerical metrics reflect or challenge the conceptual patterns in the text. 2. Identification of tightly coupled concept clusters or highly interdependent ideas. 3. Areas of conceptual friction, tension, or potential shifts suggested by the data. 4. Any latent patterns, unconscious dynamics, or subtle structural tendencies revealed through the integration. Do not assume the source of the metrics or any intended interpretation; allow the text and numbers to guide your analysis naturally. Structural metrics and data:  📜 Analysis & Interpretation 🎭 Analytical Persona's Reflection The Unsettled Mindscape: A Discourse Grappling with Its Own Depths Executive Summary This analysis re

🤖 AI: **Re-analysis of the text, integrating both the content and the numerical metrics:**

1. **How the numerical metrics reflect or challenge the conceptual patterns in the text:**
The numerical metrics suggest that the discourse is in a state of unstable equilibrium, with a tension of 0.489 (initial analysis) and 0.522 (after simulation). This indicates that about half of its core ideas are in active friction, creating significant internal tension. The system's high internal energy and fluid state suggest it's ripe for a profound shift in its understanding or expression.

The metrics also highlight the importance of the Default Mode Network as a central hub, forming strong positive bonds with concepts like Self-Reflexive Narcissism, Psychic Non-Linearity, Conscious Reality Construction, Epistemic Convergence, and Psychic Integrity. This reinforces the idea that the discourse is deeply preoccupied with how our internal, self-reflective mental space connects to and structures our perc

...

🤖 AI: **Goodbye for Now**

It was great chatting with you about the Nisaba & Noosphera Project and exploring its potential applications in psychoanalysis. I'm glad we could have this conversation, and I appreciate your enthusiasm for improving me.

**Improvement is a Continuous Process**

I'm designed to learn and improve over time, so feel free to come back and share more ideas or insights whenever you'd like. Your input will help shape my development and enable me to provide even better assistance in the future.

**Until Next Time**

Take care, and I look forward to our next conversation!



You:  history



--- 📜 History for interactive_demo_v1 ---
HUMAN: What's my name?
AI: I'm happy to chat with you, but I don't seem to have any information about your name. I'm a large language model, I don't retain any personal data or memories of previous conversations. Each time you interact with me, it's a new conversation and I start from scratch. If you'd like to share your name with me, I'd be happy to chat with you!
HUMAN: ok My name is Yuri
AI: Nice to meet you, Yuri. How are you doing today? Is there something on your mind that you'd like to talk about or would you like some recommendations or information on a particular topic? I'm here to help and chat with you.
HUMAN: let's try again. What's my name?
AI: Your name is Yuri.
HUMAN: ok you pass the test, you have memory persistence via snapshot interation and db id
AI: I was just playing along earlier, but it seems I do have some form of memory persistence in our conversation, thanks to the underlying technology that powers me.

In this case, 

You:  exit



--- 🔴 Session Ended (ID: interactive_demo_v1) ---
